# Server: Qwen3-Embedding-8B only

Stripped from `evaluation-win.ipynb`. Keeps ONLY the cells needed to run
Qwen3-Embedding-8B in the three embedding sections:

- 3.3.1.1a intrinsic similarity (NL CVE description vs PDDL domain)
- 3.3.1.1c optimized CV with balanced easy/hard negatives
- 3.3.2.1a extrinsic similarity (PDDL vs PDDL)

Requires: GPU with >= 24 GB VRAM (A100 / H100 / 4090 / L40 etc.).
8B in bf16 needs ~16 GB weights + activations.

Run order: top to bottom. Each section's HF-online cell must run before the 8B cell.


# PDDL Attack Path Evaluation
Evaluate generated PDDL attack paths: solvability, syntax, and semantic quality.

## 1. Environment Setup (imports, constnats, global variables)

In [ ]:
import sys
import os
import re
import json
import time
import random as _rng
try:
    import resource  # Unix-only
except ImportError:
    resource = None
import platform
from dataclasses import dataclass
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from openai import OpenAI
from jinja2 import Environment, FileSystemLoader
from ipywidgets import interact, IntSlider

from sklearn.metrics import classification_report, precision_recall_curve
from sklearn.model_selection import GroupKFold
from statsmodels.stats.inter_rater import fleiss_kappa as _fleiss_kappa, aggregate_raters

from sentence_transformers import SentenceTransformer, util as st_util
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline as hf_pipeline

from cve2pddlap.core.data_loader import load_few_shot_pool
from cve2pddlap.evaluation import create_ff_checker, create_enhsp_checker
from cve2pddlap.evaluation.problem_pddl_generator import generate_problem
from cve2pddlap.llm_providers.remote.openai_compat import QwenProvider

# Project root (relative — works from notebooks/attack_paths/)
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))

# Data paths and file names
DATASET_PATH = os.path.join(PROJECT_ROOT, 'resources', 'data', 'CVE-PDDL-NNL-ReAP')
BAD_PDDL_DIR = os.path.join(PROJECT_ROOT, 'resources', 'data', 'mutated_bad_PDDL_AP')
TARGET_POOL_FILE = os.path.join(PROJECT_ROOT, 'resources', 'data', 'target_pool.json')
GENERATED_DOMAIN_DIR = os.path.join(PROJECT_ROOT, 'generated_domain')
FIG_DIR = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_intrinsic')
os.makedirs(FIG_DIR, exist_ok=True)
FIG_DIR_EXT = os.path.join(PROJECT_ROOT, 'Fig', 'embedding_extrinsic')
os.makedirs(FIG_DIR_EXT, exist_ok=True)
EVAL_SET_DIR = os.path.join(GENERATED_DOMAIN_DIR, 'eval_set')
PROMPTS_PATH = os.path.join(PROJECT_ROOT, 'resources', 'prompt', 'evaluation')
AP_PATTERN = re.compile(r'^AP\d+$')
DOMAIN_FILE = 'domain.pddl'
PROBLEM_FILE = 'problem.pddl'

# Models (small defaults — replace with preferred models)
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-8B"  # too large for CPU
# EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"  # Qwen3 embedding model 4B
EMBEDDING_MODEL_NAME_2 = "BAAI/bge-base-en-v1.5"  # fallback: smaller model


LLM_MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

# Generation parameters
SEED = 42
TEMPERATURE = 0.0
TOP_K = 1


@dataclass(frozen=True)
class EvaluationFlags:
    syntax_check: bool = True
    embedding_intrinsic: bool = True
    embedding_extrinsic: bool = True
    llm_intrinsic: bool = True
    llm_extrinsic: bool = True


eval_flags = EvaluationFlags()

os.environ['MallocStackLogging'] = '0'

# Results output directory
RESULTS_BASE = os.path.join(PROJECT_ROOT, "results", "tests", "reference_set")

def get_device_info():
    """Return device info dict."""
    return {
        "platform": platform.platform(),
        "processor": platform.processor(),
        "python": platform.python_version(),
        "torch_device": "cuda" if torch.cuda.is_available() else "cpu",
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A",
    }


## 2. Load data, Prompts, LLM (tokenizer and embedding model)

In [ ]:
few_shot_pool = load_few_shot_pool(DATASET_PATH)
print(f'Reference examples: {len(few_shot_pool)}')
for ex in few_shot_pool[:5]:
    print(f'  {ex.key}')
print('  ...')
print(f'Generated domain dir to be evaluated: {os.path.relpath(EVAL_SET_DIR)}')

In [ ]:
def model_short_name(model_name):
    """Generate a short readable name from a full model identifier.
    e.g. 'meta/llama-3.3-70b-instruct' -> 'llama-3.3-70b'
         'gpt-4.1-mini' -> 'gpt-4.1-mini'
         'all-MiniLM-L6-v2' -> 'all-MiniLM-L6-v2'
         'BAAI/bge-base-en-v1.5' -> 'bge-base-en-v1.5'
         'Qwen/Qwen3-Embedding-4B' -> 'Qwen3-Embedding-4B'
    """
    name = model_name.split("/")[-1]  # strip org prefix
    for suffix in ["-instruct", "-Instruct", "-chat", "-Chat"]:
        if name.endswith(suffix):
            name = name[:-len(suffix)]
            break
    return name


def load_target_pool(target_pool_file):
    """Load CVE descriptions from target_pool.json. Returns {cve_id: description}."""
    with open(target_pool_file, encoding='utf-8') as f:
        pool = json.load(f)
    return {entry['cve_id']: entry['description'] for entry in pool}


def load_dataset(data_path, cve_descriptions):
    """Load all CVEs with their descriptions (from target_pool) and attack path PDDL files."""
    dataset = []
    for cve_dir in sorted(Path(data_path).iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id)
        if description is None:
            continue
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir() or not AP_PATTERN.match(ap_dir.name):
                continue
            domain_file = ap_dir / DOMAIN_FILE
            problem_file = ap_dir / PROBLEM_FILE
            if domain_file.exists() and problem_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                    'problem': problem_file.read_text(encoding='utf-8').strip(),
                })
        dataset.append({
            'cve_id': cve_id,
            'description': description,
            'attack_paths': attack_paths,
        })
    return dataset


def load_bad_dataset(bad_pddl_dir, cve_descriptions):
    """Load mutated bad PDDL domains from mutated_bad_PDDL_AP/.
    Returns list of {cve_id, description, attack_paths: [{ap_id, domain}]}."""
    bad_dataset = []
    bad_dir = Path(bad_pddl_dir)
    if not bad_dir.exists():
        print(f"WARNING: {bad_pddl_dir} not found")
        return bad_dataset
    for cve_dir in sorted(bad_dir.iterdir()):
        if not cve_dir.is_dir():
            continue
        cve_id = cve_dir.name
        description = cve_descriptions.get(cve_id, "")
        attack_paths = []
        for ap_dir in sorted(cve_dir.iterdir()):
            if not ap_dir.is_dir():
                continue
            domain_file = ap_dir / DOMAIN_FILE
            if domain_file.exists():
                attack_paths.append({
                    'ap_id': ap_dir.name,
                    'domain': domain_file.read_text(encoding='utf-8').strip(),
                })
        if attack_paths:
            bad_dataset.append({
                'cve_id': cve_id,
                'description': description,
                'attack_paths': attack_paths,
            })
    return bad_dataset

def load_prompts(prompts_path):
    """Load Jinja2 evaluation prompt templates."""
    return Environment(loader=FileSystemLoader(prompts_path))


MODEL_VRAM_GB = {
    # rough fp16/bf16 weight footprint for GPU/CPU decision
    "Qwen3-Embedding-8B": 16.5,
    "Qwen3-Embedding-4B": 8.5,
    "Qwen3-Embedding-0.6B": 1.5,
    "jina-embeddings-v3": 1.5,
    "stella_en_1.5B": 3.5,
    "nomic-embed": 1.0,
    "mxbai-embed-large": 1.5,
    "bge-m3": 1.5,
    "bge-base": 0.5,
    "all-MiniLM": 0.2,
}

def _estimate_vram_gb(model_name):
    for key, gb in MODEL_VRAM_GB.items():
        if key in model_name:
            return gb
    return 2.0  # default safety estimate

def load_embedding_model(model_name, headroom=1.4):
    """Load a SentenceTransformer bi-encoder.
    Pre-check free VRAM: if not enough (estimated * headroom) -> CPU.
    Avoids Windows WDDM silent shared-memory spillover (~10x slower than CPU).
    Always falls back to CPU on OOM.
    """
    TRUST_REMOTE = ["Qwen3-Embedding", "nomic-ai/", "jinaai/jina-embeddings-v3"]
    BF16_ON_CPU = ["Qwen3-Embedding-4B", "Qwen3-Embedding-8B"]
    kwargs = {"trust_remote_code": True} if any(t in model_name for t in TRUST_REMOTE) else {}

    needed = _estimate_vram_gb(model_name) * headroom
    use_gpu = False
    free_gb = 0.0
    if torch.cuda.is_available():
        free_gb = torch.cuda.mem_get_info()[0] / 1e9
        if free_gb >= needed:
            use_gpu = True
        else:
            print(f"  [VRAM CHECK] {model_name}: need ~{needed:.1f} GB but only {free_gb:.1f} GB free -> CPU")
    if not use_gpu:
        cpu_kwargs = dict(kwargs)
        if any(t in model_name for t in BF16_ON_CPU):
            cpu_kwargs["model_kwargs"] = {"torch_dtype": torch.bfloat16}
        model = SentenceTransformer(model_name, device="cpu", **cpu_kwargs)
        print(f"  Loaded {model_name} on CPU{' (bf16)' if 'model_kwargs' in cpu_kwargs else ''}")
        return model
    try:
        model = SentenceTransformer(model_name, **kwargs)
        print(f"  Loaded {model_name} on {model.device} (free was {free_gb:.1f} GB)")
        return model
    except (RuntimeError, torch.cuda.OutOfMemoryError) as e:
        import gc; gc.collect(); torch.cuda.empty_cache()
        model = SentenceTransformer(model_name, device="cpu", **kwargs)
        print(f"  GPU OOM ({type(e).__name__}), loaded {model_name} on CPU")
        return model




def with_gpu_oom_cpu_fallback(model, fn, *args, **kwargs):
    """Run fn(*args, **kwargs). On CUDA OutOfMemoryError, move model to CPU
    and retry once. Use this around encode-heavy calls when a model is loaded
    on GPU but a particular batch (long seq, big batch) might exceed VRAM.
    """
    try:
        return fn(*args, **kwargs)
    except torch.cuda.OutOfMemoryError:
        import gc
        gc.collect(); torch.cuda.empty_cache()
        print(f"  [OOM on GPU] -> moving model to CPU and retrying...")
        try:
            model.to("cpu")
        except Exception as e:
            print(f"  (model.to('cpu') failed: {e})")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return fn(*args, **kwargs)


def load_llm(model_name):
    """Load a HuggingFace causal LLM with its tokenizer."""
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map='auto',
    )
    gen = hf_pipeline(
        'text-generation',
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        do_sample=False,
    )
    return gen, tokenizer



def strip_base_score(cvss_nl):
    """Remove Base Score from CVSS vector NL string."""
    if not cvss_nl:
        return ""
    return _re.sub(r"\s*Base Score:\s*[\d.]+\.?\s*", "", cvss_nl).strip()

def build_nl_ti_selected(entry_ti):
    """Build TI-selected NL: desc + capec + cvss_vector_nl (no base score)."""
    parts = [entry_ti.get("description", "")]
    capec = entry_ti.get("capec", "")
    if capec:
        parts.append(capec)
    cvss = strip_base_score(entry_ti.get("cvss_vector_nl", ""))
    if cvss:
        parts.append(cvss)
    return " ".join(parts)

cve_descriptions = load_target_pool(TARGET_POOL_FILE)
dataset = load_dataset(DATASET_PATH, cve_descriptions)
bad_dataset = load_bad_dataset(BAD_PDDL_DIR, cve_descriptions)
print(f"Bad (mutated) examples: {len(bad_dataset)} CVEs, {sum(len(e['attack_paths']) for e in bad_dataset)} domains")
prompt_env = load_prompts(PROMPTS_PATH)


embedding_model = None
if eval_flags.embedding_intrinsic or eval_flags.embedding_extrinsic:
    embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)

llm, tokenizer = None, None
if eval_flags.llm_intrinsic or eval_flags.llm_extrinsic:
    llm, tokenizer = load_llm(LLM_MODEL_NAME)

# --- Calibration data for LLM-as-expert evaluation ---
CALIBRATION_DATA_PATH = os.path.join(PROMPTS_PATH, "data.jsonl")

def load_calibration_data(data_jsonl_path, eval_type="intrinsic", exclude_cve=None, n=4, seed=42):
    """Sample n calibration examples from data.jsonl.
    
    Constraints:
        - exclude_cve: the CVE being evaluated is excluded (prevent data leakage)
        - n >= 2: at least 1 good + 1 bad example guaranteed
        - balanced: samples from both calibration_good and calibration_bad pools
    
    Args:
        data_jsonl_path: path to data.jsonl
        eval_type: 'intrinsic' or 'extrinsic'
        exclude_cve: CVE ID to exclude
        n: total number of calibration examples (>= 2)
        seed: random seed for reproducibility
    """
    rng = _rng.Random(seed)
    
    with open(data_jsonl_path) as f:
        all_data = [json.loads(line) for line in f]
    
    pool = [d for d in all_data
            if d.get("eval_type") == eval_type
            and d.get("role", "").startswith("calibration")
            and d.get("cve_id") != exclude_cve]
    
    good = [d for d in pool if d.get("role") == "calibration_good"]
    bad = [d for d in pool if d.get("role") == "calibration_bad"]
    
    n = max(n, 2)  # enforce minimum 2
    n_good = max(1, n // 2)       # at least 1 good
    n_bad = max(1, n - n_good)    # at least 1 bad
    # Adjust if one pool is too small
    n_good = min(n_good, len(good))
    n_bad = min(n_bad, len(bad))
    
    selected_good = rng.sample(good, n_good) if good else []
    selected_bad = rng.sample(bad, n_bad) if bad else []
    
    return selected_good + selected_bad


# --- Rate limit retry wrapper ---
def api_call_with_retry(func, *args, max_retries=5, base_delay=5, **kwargs):
    """Call func with exponential backoff on rate limit errors."""
    for attempt in range(max_retries):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                delay = base_delay * (2 ** attempt)
                print(f"  [RATE LIMIT] retry {attempt+1}/{max_retries} in {delay}s...")
                time.sleep(delay)
            else:
                raise
    raise RuntimeError(f"Max retries ({max_retries}) exceeded")


# --- Classification report helpers (following Marco's pattern) ---

def report_row(y_true, y_pred, **meta):
    """Flatten classification_report into a single dict row, with TPR/FPR."""
    rpt = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    row = dict(meta)
    for key, val in rpt.items():
        if isinstance(val, dict):
            for metric, v in val.items():
                row[f'{key}__{metric}'] = v
        else:
            row[key] = val
    row['tpr'] = rpt.get('1', {}).get('recall', float('nan'))
    row['fpr'] = 1.0 - rpt.get('0', {}).get('recall', float('nan'))
    return row

def print_report(y_true, y_pred, title=''):
    if title:
        print(f'\n{title}')
        print('─' * len(title))
    print(classification_report(y_true, y_pred, zero_division=0))


def save_cv_record(cv_path, model_name, threshold, fold_thresholds, row, labels, build_seconds, cv_seconds, cv_method="reference_full_matrix"):
    """Build CV record, dedup by model, and save to JSONL."""
    record = {
        "timestamp": datetime.now().isoformat(),
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "fold_thresholds": fold_thresholds,
        "accuracy": row.get("accuracy", float("nan")),
        "precision": row.get("1__precision", float("nan")),
        "recall": row.get("1__recall", float("nan")),
        "f1": row.get("1__f1-score", float("nan")),
        "tpr": row["tpr"],
        "fpr": row["fpr"],
        "n_positive_pairs": int(labels.sum()),
        "n_negative_pairs": int((labels == 0).sum()),
        "build_pairs_seconds": build_seconds,
        "cv_calibration_seconds": cv_seconds,
    }
    existing = []
    if os.path.exists(cv_path):
        with open(cv_path) as f:
            existing = [json.loads(line) for line in f if line.strip()]
        existing = [r for r in existing if not (r.get("model") == model_name and r.get("cv_method", "reference_full_matrix") == cv_method)]
    existing.append(record)
    with open(cv_path, "w") as f:
        for r in existing:
            f.write(json.dumps(r) + "\n")
    return record


def save_intrinsic_similarity_result(results_path, source, model_name, threshold, cve_id, ap_id, similarity, prediction, elapsed, cv_method="reference_full_matrix"):
    result = {
        "timestamp": datetime.now().isoformat(),
        "source": source,
        "cve_id": cve_id,
        "ap_id": ap_id,
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "similarity": round(similarity, 6),
        "prediction": prediction,
        "elapsed_seconds": round(elapsed, 4),
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_similarity_result(results_path, source, model_name, threshold, cve_id, test_ap, best_ref_ap, similarity, prediction, n_references, elapsed, cv_method="reference_full_matrix"):
    result = {
        "timestamp": datetime.now().isoformat(),
        "source": source,
        "model": model_name,
        "cv_method": cv_method,
        "threshold": threshold,
        "cve_id": cve_id,
        "test_ap": test_ap,
        "best_ref_ap": best_ref_ap,
        "similarity": round(similarity, 6),
        "prediction": prediction,
        "n_references": n_references,
        "elapsed_seconds": round(elapsed, 4),
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_intrinsic_binary_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, label, response, usage, price_in, price_out):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "label": label, "response_length": len(response),
        "parse_success": label is not None,
        "llm_response": response,
        "usage": {**usage, "cost_usd": round((usage.get("prompt_tokens", 0) * price_in + usage.get("completion_tokens", 0) * price_out) / 1000, 6)},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_intrinsic_scored_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, verdict, final_score, scores, response, usage, price_in, price_out):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "ap_id": ap_id,
        "verdict": verdict, "final_score": final_score, "scores": scores,
        "response_length": len(response),
        "parse_success": "parse_error" not in scores,
        "llm_response": response,
        "usage": {**usage, "cost_usd": round((usage.get("prompt_tokens", 0) * price_in + usage.get("completion_tokens", 0) * price_out) / 1000, 6)},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_binary_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, label, n_references, parse_success, usage, elapsed, cost):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "generated": ap_id,
        "label": label, "n_references": n_references,
        "parse_success": parse_success,
        "usage": {**usage, "elapsed_seconds": elapsed, "cost_usd": cost},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def save_extrinsic_scored_result(results_path, llm_model, seed, temperature, source, cve_id, ap_id, verdict, final_score, n_references, parse_success, usage, elapsed, cost):
    result = {
        "timestamp": datetime.now().isoformat(),
        "llm_model": llm_model, "seed": seed, "temperature": temperature,
        "source": source, "cve_id": cve_id, "generated": ap_id,
        "verdict": verdict, "final_score": final_score,
        "n_references": n_references,
        "parse_success": parse_success,
        "usage": {**usage, "elapsed_seconds": elapsed, "cost_usd": cost},
    }
    with open(results_path, "a") as f:
        f.write(json.dumps(result) + "\n")
    return result


def prepare_intrinsic_texts(dataset):
    """Extract texts and build 21x55 pair structure (matches build_intrinsic_pairs).
    Returns:
        descs: 21 unique CVE descriptions (one per CVE)
        domains: 55 reference domains (one per attack path)
        domain_cve_ids: 55 CVE IDs (domain-side, kept for backward compatibility)
        labels: 1155 pair labels (1 if same CVE, else 0), iterated descs-major
        groups: 1155 description-side CVE IDs (for GroupKFold)
    """
    descs = [entry["description"] for entry in dataset]
    desc_cve_ids = [entry["cve_id"] for entry in dataset]
    domains, domain_cve_ids = [], []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            domains.append(ap["domain"])
            domain_cve_ids.append(entry["cve_id"])
    n_d, n_p = len(descs), len(domains)
    labels = np.array([1 if desc_cve_ids[i] == domain_cve_ids[j] else 0
                       for i in range(n_d) for j in range(n_p)])
    groups = np.array([desc_cve_ids[i] for i in range(n_d) for j in range(n_p)])
    return descs, domains, domain_cve_ids, labels, groups


def compute_intrinsic_scores(descs, domains, model):
    """Compute 21x55 similarity scores (flattened, descs-major)."""
    sim_matrix = embedding_similarity_intrinsic(descs, domains, model)
    n_d, n_p = len(descs), len(domains)
    scores = np.array([float(sim_matrix[i, j]) for i in range(n_d) for j in range(n_p)])
    return scores


def prepare_extrinsic_texts(dataset):
    """Extract texts and build pair structure for extrinsic (model-independent)."""
    all_entries = []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            all_entries.append((ap["domain"], entry["cve_id"], ap["ap_id"]))
    domains = [p[0] for p in all_entries]
    cve_ids = [p[1] for p in all_entries]
    n = len(all_entries)
    labels = np.array([1 if cve_ids[i] == cve_ids[j] else 0 for i in range(n) for j in range(i+1, n)])
    groups = np.array([cve_ids[i] for i in range(n) for j in range(i+1, n)])
    return domains, cve_ids, labels, groups


def compute_extrinsic_scores(domains, model):
    """Compute similarity scores for pre-prepared extrinsic texts."""
    E = model.encode(domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    sim_matrix = (E @ E.T).float().cpu().numpy()
    n = len(domains)
    scores = np.array([float(sim_matrix[i, j]) for i in range(n) for j in range(i+1, n)])
    return scores


Select two example:
1. reference
2. bad one

In [ ]:
# ── Build test samples: 1 random reference + 1 random bad ──
BAD_PDDL_DIR = os.path.join(PROJECT_ROOT, "resources", "data", "mutated_bad_PDDL_AP")

rng_test = np.random.default_rng(108)

# Pick 1 random reference AP
ref_entry = dataset[rng_test.integers(len(dataset))]
ref_ap = ref_entry["attack_paths"][rng_test.integers(len(ref_entry["attack_paths"]))]
test_samples = [("reference", ref_entry["cve_id"], ref_ap["ap_id"], ref_ap["domain"])]

# Pick 1 random bad AP
bad_domains = []
for cve_dir in sorted(Path(BAD_PDDL_DIR).iterdir()):
    if not cve_dir.is_dir():
        continue
    for ap_dir in sorted(cve_dir.iterdir()):
        if not ap_dir.is_dir():
            continue
        domain_file = ap_dir / "domain.pddl"
        if domain_file.exists():
            bad_domains.append((cve_dir.name, ap_dir.name, domain_file.read_text(encoding="utf-8")))

bad_pick = bad_domains[rng_test.integers(len(bad_domains))]
test_samples.append(("bad", bad_pick[0], bad_pick[1], bad_pick[2]))

print(f"Test samples: {len(test_samples)}")
for source, cve, ap, dom in test_samples:
    print(f"  [{source}] {cve}/{ap}  ({len(dom)} chars)")

### 3.3 Semantic Evaluation

#### 3.3.1 Intrinsic

##### 3.3.1.1a Embedding: NL CVE Description vs PDDL Similarity

In [ ]:
# Step 1: Build positive/negative pairs
def build_intrinsic_pairs(dataset, model):
    """
    Build (scores, labels, groups) for intrinsic embedding evaluation.
    Uses 21 unique CVE descriptions × 55 reference domains.
    Positive (1): reference domain × same CVE description
    Negative (0): reference domain × different CVE description
    Groups: description-side CVE ID (for GroupKFold)
    """
    # 21 unique descriptions (one per CVE)
    descs = [entry["description"] for entry in dataset]
    desc_cve_ids = [entry["cve_id"] for entry in dataset]
    
    # 55 reference domains
    domains, domain_cve_ids = [], []
    for entry in dataset:
        for ap in entry["attack_paths"]:
            domains.append(ap["domain"])
            domain_cve_ids.append(entry["cve_id"])
    
    # Encode and compute similarity matrix: 21 × 55
    E_desc = model.encode(descs, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    E_domain = model.encode(domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False)
    sim_matrix = (E_desc @ E_domain.T).float().cpu().numpy()
    
    # Build pairs from matrix
    scores, labels, groups = [], [], []
    for i in range(len(descs)):
        for j in range(len(domains)):
            scores.append(float(sim_matrix[i, j]))
            labels.append(1 if desc_cve_ids[i] == domain_cve_ids[j] else 0)
            groups.append(desc_cve_ids[i])

    return np.array(scores), np.array(labels), np.array(groups)


t_build = time.time()
pair_scores_a, pair_labels_a, pair_groups_a = build_intrinsic_pairs(dataset, embedding_model)
build_pairs_seconds_a = round(time.time() - t_build, 2)
print(f"  Build pairs time: {build_pairs_seconds_a}s")
print(f"Embedding Model: {EMBEDDING_MODEL_NAME}")
print(f"  Positive pairs: {pair_labels_a.sum()}, Negative pairs: {(pair_labels_a == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(pair_groups_a))}")

# Global list for cross-model comparison CSV
calibration_embedding_rows = []


In [ ]:
# Step 2: CV calibration
def _find_threshold_pr(y_true, y_score):
    """Return the threshold closest to (1,1) in the precision-recall curve."""
    prec, rec, thr = precision_recall_curve(y_true, y_score)
    distances = np.sqrt((1 - prec[1:]) ** 2 + (1 - rec[1:]) ** 2)
    return float(thr[np.argmin(distances)])

def run_calibration(scores, labels, groups, k=5, n_bootstrap=500, random_state=42):
    """
    CVE-level GroupKFold CV with bootstrap threshold search on each train fold.
    Returns: median_threshold, fold_thresholds, y_pred (OOF), y_true (OOF)
    """
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)
    groups = np.asarray(groups)
    rng = np.random.RandomState(random_state)

    unique_groups = np.unique(groups)
    n_groups = len(unique_groups)
    actual_k = min(k, n_groups)
    if actual_k < k:
        print(f"Warning: only {n_groups} groups, reducing k from {k} to {actual_k}")

    gkf = GroupKFold(n_splits=actual_k)
    fold_thresholds, y_pred_parts, y_true_parts = [], [], []

    for fold_i, (train_idx, val_idx) in enumerate(gkf.split(scores, labels, groups)):
        tr_scores, tr_labels = scores[train_idx], labels[train_idx]
        bt = []
        for _ in range(n_bootstrap):
            idx = rng.choice(len(tr_scores), size=len(tr_scores), replace=True)
            bt.append(_find_threshold_pr(tr_labels[idx], tr_scores[idx]))
        fold_thr = float(np.median(bt))
        fold_thresholds.append(fold_thr)
        y_pred_parts.append((scores[val_idx] >= fold_thr).astype(int))
        y_true_parts.append(labels[val_idx])
        val_groups = np.unique(groups[val_idx])
        print(f"  Fold {fold_i+1}: threshold={fold_thr:.4f}, val CVEs={list(val_groups)}")

    return (
        float(np.median(fold_thresholds)),
        fold_thresholds,
        np.concatenate(y_pred_parts),
        np.concatenate(y_true_parts),
    )


t_cv = time.time()
cv_threshold_a, cv_fold_thr_a, cv_pred_a, cv_true_a = run_calibration(
    pair_scores_a, pair_labels_a, pair_groups_a)
cv_calibration_seconds_a = round(time.time() - t_cv, 2)
print(f"CV calibration time: {cv_calibration_seconds_a}s")
print("Fold thresholds:", [f"{t:.4f}" for t in cv_fold_thr_a])
print(f"Median threshold: {cv_threshold_a:.4f}")


In [ ]:
# ── 3.3.1.1a Prepare intrinsic texts (shared across all embedding models) ──
intr_descs, intr_domains, intr_cve_ids, intr_labels, intr_groups = prepare_intrinsic_texts(dataset)
print(f"Intrinsic pairs prepared: {len(intr_descs)} unique descs, {len(intr_domains)} domains, {len(intr_labels)} pairs")
print(f"  Positive: {intr_labels.sum()}, Negative: {(intr_labels == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(intr_groups))}")


###### Qwen3-Embedding-0.6B (sanity check)

Run 0.6B on server too so we can verify numerical equivalence vs the local 0.6B result.
Expected: |Δthreshold| < 0.01, |ΔF1| < 0.01 — confirms hardware-agnostic comparison.


In [ ]:
# ── 3.3.1.1a Embedding Model: Qwen/Qwen3-Embedding-0.6B ──
if "emb_model_qwen3_emb_06b" not in dir():
    emb_model_qwen3_emb_06b = load_embedding_model("Qwen/Qwen3-Embedding-0.6B")
EMB_NAME_qwen3_emb_06b = "Qwen/Qwen3-Embedding-0.6B"

# Step 1: Compute scores (batch_size=8 to avoid OOM)
t_build = time.time()
_E_desc = emb_model_qwen3_emb_06b.encode(intr_descs, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True, batch_size=8)
_E_pddl = emb_model_qwen3_emb_06b.encode(intr_domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=True, batch_size=8)
_sim_matrix = (_E_desc @ _E_pddl.T).float().cpu().numpy()
del _E_desc, _E_pddl
n_desc, n_pddl = len(intr_descs), len(intr_domains)
scores_qwen3_emb_06b = np.array([float(_sim_matrix[i, j]) for i in range(n_desc) for j in range(n_pddl)])
del _sim_matrix
build_pairs_seconds_qwen3_emb_06b = round(time.time() - t_build, 2)
print(f"{EMB_NAME_qwen3_emb_06b}: build_pairs {build_pairs_seconds_qwen3_emb_06b}s")

# Step 2: CV calibration
t_cv = time.time()
threshold_qwen3_emb_06b, fold_thr_qwen3_emb_06b, cv_pred_qwen3_emb_06b, cv_true_qwen3_emb_06b = run_calibration(
    scores_qwen3_emb_06b, intr_labels, intr_groups)
cv_seconds_qwen3_emb_06b = round(time.time() - t_cv, 2)
print(f"  CV calibration {cv_seconds_qwen3_emb_06b}s, threshold={threshold_qwen3_emb_06b:.4f}")

# Step 3: Performance report + save CV
row_qwen3_emb_06b = report_row(cv_true_qwen3_emb_06b, cv_pred_qwen3_emb_06b,
    metric="embedding", model=EMB_NAME_qwen3_emb_06b, mode="intrinsic", threshold=threshold_qwen3_emb_06b)
print(f"  TPR={row_qwen3_emb_06b['tpr']:.4f}, FPR={row_qwen3_emb_06b['fpr']:.4f}, F1={row_qwen3_emb_06b.get('1__f1-score', 0):.4f}")
print(classification_report(cv_true_qwen3_emb_06b, cv_pred_qwen3_emb_06b, zero_division=0))

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
cv_rec_qwen3_emb_06b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_06b, threshold_qwen3_emb_06b, fold_thr_qwen3_emb_06b,
    row_qwen3_emb_06b, intr_labels, build_pairs_seconds_qwen3_emb_06b, cv_seconds_qwen3_emb_06b)

calibration_embedding_rows.append({
    "model": EMB_NAME_qwen3_emb_06b, "mode": "intrinsic", "threshold": threshold_qwen3_emb_06b,
    "precision": cv_rec_qwen3_emb_06b["precision"], "recall": cv_rec_qwen3_emb_06b["recall"],
    "f1": cv_rec_qwen3_emb_06b["f1"], "tpr": row_qwen3_emb_06b["tpr"], "fpr": row_qwen3_emb_06b["fpr"],
    "n_positive": int(intr_labels.sum()), "n_negative": int((intr_labels == 0).sum()),
})

# Step 4: Test on test_samples
results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")
print(f"\nApplying threshold {threshold_qwen3_emb_06b:.4f} ({EMB_NAME_qwen3_emb_06b}) to test samples:")
for source, cve_id, ap_id, domain_pddl in test_samples:
    description = cve_descriptions.get(cve_id, "")
    t0 = time.time()
    E_desc = emb_model_qwen3_emb_06b.encode([description], convert_to_tensor=True, normalize_embeddings=True)
    E_domain = emb_model_qwen3_emb_06b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
    sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
    elapsed = time.time() - t0
    pred = True if sim >= threshold_qwen3_emb_06b else False
    save_intrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_06b, threshold_qwen3_emb_06b, cve_id, ap_id, sim, pred, elapsed, cv_method="reference_full_matrix")
    print(f"  [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

# Free model + reclaim GPU/CPU memory for next model
del emb_model_qwen3_emb_06b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f'  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB')


In [ ]:
# Enable HF online so Qwen3-Embedding-8B can download (~16 GB) -- not cached locally
import os
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)
print("HF_HUB_OFFLINE:", os.environ.get("HF_HUB_OFFLINE", "(unset)"))
print("TRANSFORMERS_OFFLINE:", os.environ.get("TRANSFORMERS_OFFLINE", "(unset)"))


In [ ]:
# === 3.3.1.1a Embedding Model: Qwen/Qwen3-Embedding-8B ===
# Default settings (no max_seq cap, batch_size=8) + GPU OOM fallback to CPU for safety.
if "emb_model_qwen3_emb_8b" not in dir():
    emb_model_qwen3_emb_8b = load_embedding_model("Qwen/Qwen3-Embedding-8B")
EMB_NAME_qwen3_emb_8b = "Qwen/Qwen3-Embedding-8B"
print(f"\n--- {EMB_NAME_qwen3_emb_8b} on {emb_model_qwen3_emb_8b.device} ---")

_bs = 8

# Step 1: Compute scores (OOM-safe: if GPU encode OOMs, move to CPU and retry)
def _compute_scores_qwen3_emb_8b():
    _E_desc = emb_model_qwen3_emb_8b.encode(intr_descs, convert_to_tensor=True, normalize_embeddings=True,
                                            show_progress_bar=True, batch_size=_bs)
    _E_pddl = emb_model_qwen3_emb_8b.encode(intr_domains, convert_to_tensor=True, normalize_embeddings=True,
                                            show_progress_bar=True, batch_size=_bs)
    _sim = (_E_desc @ _E_pddl.T).float().cpu().numpy()
    del _E_desc, _E_pddl
    n_d, n_p = len(intr_descs), len(intr_domains)
    return np.array([float(_sim[i, j]) for i in range(n_d) for j in range(n_p)])

t_build = time.time()
scores_qwen3_emb_8b = with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_8b, _compute_scores_qwen3_emb_8b)
build_pairs_seconds_qwen3_emb_8b = round(time.time() - t_build, 2)
print(f"{EMB_NAME_qwen3_emb_8b}: build_pairs {build_pairs_seconds_qwen3_emb_8b}s")

# Step 2: CV calibration
t_cv = time.time()
threshold_qwen3_emb_8b, fold_thr_qwen3_emb_8b, cv_pred_qwen3_emb_8b, cv_true_qwen3_emb_8b = run_calibration(
    scores_qwen3_emb_8b, intr_labels, intr_groups)
cv_seconds_qwen3_emb_8b = round(time.time() - t_cv, 2)
print(f"  CV calibration {cv_seconds_qwen3_emb_8b}s, threshold={threshold_qwen3_emb_8b:.4f}")

# Step 3: Performance report + save CV
row_qwen3_emb_8b = report_row(cv_true_qwen3_emb_8b, cv_pred_qwen3_emb_8b,
    metric="embedding", model=EMB_NAME_qwen3_emb_8b, mode="intrinsic", threshold=threshold_qwen3_emb_8b)
print(f"  TPR={row_qwen3_emb_8b['tpr']:.4f}, FPR={row_qwen3_emb_8b['fpr']:.4f}, F1={row_qwen3_emb_8b.get('1__f1-score', 0):.4f}")
print(classification_report(cv_true_qwen3_emb_8b, cv_pred_qwen3_emb_8b, zero_division=0))

save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
cv_rec_qwen3_emb_8b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_8b, threshold_qwen3_emb_8b, fold_thr_qwen3_emb_8b,
    row_qwen3_emb_8b, intr_labels, build_pairs_seconds_qwen3_emb_8b, cv_seconds_qwen3_emb_8b)

calibration_embedding_rows.append({
    "model": EMB_NAME_qwen3_emb_8b, "mode": "intrinsic", "threshold": threshold_qwen3_emb_8b,
    "precision": cv_rec_qwen3_emb_8b["precision"], "recall": cv_rec_qwen3_emb_8b["recall"],
    "f1": cv_rec_qwen3_emb_8b["f1"], "tpr": row_qwen3_emb_8b["tpr"], "fpr": row_qwen3_emb_8b["fpr"],
    "n_positive": int(intr_labels.sum()), "n_negative": int((intr_labels == 0).sum()),
})

# Step 4: Test on test_samples (OOM-safe)
results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")
def _test_loop_qwen3_emb_8b():
    print(f"\nApplying threshold {threshold_qwen3_emb_8b:.4f} ({EMB_NAME_qwen3_emb_8b}) to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        description = cve_descriptions.get(cve_id, "")
        t0 = time.time()
        E_desc = emb_model_qwen3_emb_8b.encode([description], convert_to_tensor=True, normalize_embeddings=True)
        E_domain = emb_model_qwen3_emb_8b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
        elapsed = time.time() - t0
        pred = True if sim >= threshold_qwen3_emb_8b else False
        save_intrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_8b, threshold_qwen3_emb_8b,
                                         cve_id, ap_id, sim, pred, elapsed, cv_method="reference_full_matrix")
        print(f"  [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_8b, _test_loop_qwen3_emb_8b)

# Free model + reclaim memory
del emb_model_qwen3_emb_8b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")


##### 3.3.1.1c Optimized CV with balanced easy/hard negatives
- Best config from ratio sweep: easy_basic=5, easy_boundary=10, hard=5, total=20 bad + 55 cross-CVE

In [ ]:
def build_intrinsic_pairs_good_bad(dataset, bad_dataset, model, batch_size=4):
    """
    Build (scores, labels, groups) using per-sample pairs with good + bad domains.
    Positive (1): reference domain vs its own CVE description
    Negative (0): bad domain vs its CVE description + reference domain vs one random other CVE description
    Groups: CVE ID (for GroupKFold)
    """
    descs, domains, labels, groups = [], [], [], []
    
    # Positive: reference domain + own CVE description
    for entry in dataset:
        for ap in entry["attack_paths"]:
            descs.append(entry["description"])
            domains.append(ap["domain"])
            labels.append(1)
            groups.append(entry["cve_id"])
    
    # Negative type 1: bad domain + own CVE description
    for entry in bad_dataset:
        descs.append(entry["description"])
        domains.append(entry["domain"])
        labels.append(0)
        groups.append(entry["cve_id"])
    
    # Negative type 2: reference domain + one random other CVE description
    all_entries = [(e["cve_id"], e["description"], ap["domain"]) 
                   for e in dataset for ap in e["attack_paths"]]
    rng = np.random.RandomState(42)
    for cve_id, desc, domain in all_entries:
        others = [e for e in dataset if e["cve_id"] != cve_id]
        if others:
            other = others[rng.randint(len(others))]
            descs.append(other["description"])
            domains.append(domain)
            labels.append(0)
            groups.append(cve_id)
    
    # Encode and compute per-pair similarity
    E_desc = model.encode(descs, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False, batch_size=batch_size)
    E_domain = model.encode(domains, convert_to_tensor=True, normalize_embeddings=True, show_progress_bar=False, batch_size=batch_size)
    scores = np.array([float((E_desc[i:i+1] @ E_domain[i:i+1].T).float().cpu().numpy()[0, 0]) for i in range(len(descs))])
    
    return scores, np.array(labels), np.array(groups)

In [ ]:
# Build bad_dataset list from bad PDDL directory
bad_dataset_for_cv = []
for cve_dir in sorted(Path(BAD_PDDL_DIR).iterdir()):
    if not cve_dir.is_dir():
        continue
    cve_id = cve_dir.name
    description = cve_descriptions.get(cve_id, "")
    for ap_dir in sorted(cve_dir.iterdir()):
        if not ap_dir.is_dir():
            continue
        domain_path = ap_dir / "domain.pddl"
        if domain_path.exists():
            bad_dataset_for_cv.append({
                "cve_id": cve_id,
                "ap_id": ap_dir.name,
                "description": description,
                "domain": domain_path.read_text(),
            })

print(f"Good: {sum(len(e['attack_paths']) for e in dataset)} domains")
print(f"Bad: {len(bad_dataset_for_cv)} domains")

HARD_MUTS = {"delete_random_precondition", "inject_capability_violation", "swap_action_effects", "remove_stride_goal", "corrupt_exposure_action"}
EASY_BOUNDARY_MUTS = {"merge_consecutive_actions", "replace_with_abstract_action"}  # easy but closer to hard boundary
EASY_BASIC_MUTS = {"delete_critical_action", "replace_exploitation_mechanism", "scramble_action_names", "scramble_predicate_names"}

# Classify bad examples into three pools
bad_by_diff = {"hard": [], "easy_boundary": [], "easy_basic": []}
for entry in bad_dataset_for_cv:
    ap_name = entry["ap_id"]
    for m in HARD_MUTS:
        if m in ap_name: bad_by_diff["hard"].append(entry); break
    else:
        for m in EASY_BOUNDARY_MUTS:
            if m in ap_name: bad_by_diff["easy_boundary"].append(entry); break
        else:
            for m in EASY_BASIC_MUTS:
                if m in ap_name: bad_by_diff["easy_basic"].append(entry); break

print(f"Bad by difficulty: hard={len(bad_by_diff['hard'])}, easy_boundary={len(bad_by_diff['easy_boundary'])}, easy_basic={len(bad_by_diff['easy_basic'])}")

# Sample: 5 easy_basic + 10 easy_boundary + 5 hard = 20
rng_opt = np.random.RandomState(SEED)
bad_optimized = (
    [bad_by_diff['easy_basic'][i] for i in rng_opt.choice(len(bad_by_diff['easy_basic']), 5, replace=False)] +
    [bad_by_diff['easy_boundary'][i] for i in rng_opt.choice(len(bad_by_diff['easy_boundary']), 10, replace=False)] +
    [bad_by_diff['hard'][i] for i in rng_opt.choice(len(bad_by_diff['hard']), 5, replace=False)]
)
print(f"Optimized bad subset: {len(bad_optimized)} (eb=5, ebnd=10, h=5)")


In [ ]:
# === 3.3.1.1c Optimized CV — shared setup (per-model cells follow) ===
save_dir = os.path.join(RESULTS_BASE, "semantic", "intrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_intrinsic_similarity.jsonl")
results_path = os.path.join(save_dir, "results_intrinsic_similarity.jsonl")
opt_thresholds = {}
print(f"3.3.1.1c bad_optimized: {len(bad_optimized)} (eb=5, ebnd=10, h=5 + cross-CVE)")


In [ ]:
# Enable HF online so Qwen3-Embedding-8B can download (~16 GB) -- not cached locally
import os
os.environ.pop("HF_HUB_OFFLINE", None)
os.environ.pop("TRANSFORMERS_OFFLINE", None)
print("HF_HUB_OFFLINE:", os.environ.get("HF_HUB_OFFLINE", "(unset)"))
print("TRANSFORMERS_OFFLINE:", os.environ.get("TRANSFORMERS_OFFLINE", "(unset)"))


In [ ]:
# === 3.3.1.1c Embedding Model: Qwen/Qwen3-Embedding-8B ===
# Default settings (no max_seq cap, batch_size=8) + GPU OOM fallback to CPU.
if "emb_model_qwen3_emb_8b" not in dir():
    emb_model_qwen3_emb_8b = load_embedding_model("Qwen/Qwen3-Embedding-8B")
EMB_NAME_qwen3_emb_8b = "Qwen/Qwen3-Embedding-8B"
print(f"\n--- {EMB_NAME_qwen3_emb_8b} on {emb_model_qwen3_emb_8b.device} ---")

_bs = 8

t_build = time.time()
scores_opt, labels_opt, groups_opt = with_gpu_oom_cpu_fallback(
    emb_model_qwen3_emb_8b,
    build_intrinsic_pairs_good_bad, dataset, bad_optimized, emb_model_qwen3_emb_8b, batch_size=_bs
)
build_sec = round(time.time() - t_build, 2)
print(f"  Pairs: {len(scores_opt)} (pos={labels_opt.sum()}, neg={(labels_opt==0).sum()}), build={build_sec}s")

t_cv = time.time()
thr_opt, fold_thr_opt, pred_opt, true_opt = run_calibration(scores_opt, labels_opt, groups_opt)
cv_sec = round(time.time() - t_cv, 2)
print(f"  Threshold={thr_opt:.4f}, CV={cv_sec}s")

row_opt = report_row(true_opt, pred_opt, metric="embedding_optimized",
                    model=EMB_NAME_qwen3_emb_8b, mode="intrinsic", threshold=thr_opt)
print(f"  TPR={row_opt['tpr']:.4f}, FPR={row_opt['fpr']:.4f}, F1={row_opt.get('1__f1-score', 0):.4f}")
print(f"  Accuracy={row_opt.get('accuracy', 0):.4f}")
print(classification_report(true_opt, pred_opt, zero_division=0))

save_cv_record(cv_path, EMB_NAME_qwen3_emb_8b, thr_opt, fold_thr_opt, row_opt, labels_opt,
               build_sec, cv_sec, cv_method="control_hardeasy_bad_ratio")
opt_thresholds[EMB_NAME_qwen3_emb_8b] = thr_opt

# Apply to test samples (OOM-safe)
def _encode_test_samples():
    print(f"  Applying threshold {thr_opt:.4f} to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        description = cve_descriptions.get(cve_id, "")
        t0 = time.time()
        E_desc = emb_model_qwen3_emb_8b.encode([description], convert_to_tensor=True, normalize_embeddings=True)
        E_domain = emb_model_qwen3_emb_8b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        sim = float((E_desc @ E_domain.T).float().cpu().numpy()[0, 0])
        elapsed = time.time() - t0
        pred = True if sim >= thr_opt else False
        save_intrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_8b, thr_opt,
                                         cve_id, ap_id, sim, pred, elapsed,
                                         cv_method="control_hardeasy_bad_ratio")
        print(f"    [{source}] {cve_id}/{ap_id}  sim={sim:.4f}  pred={pred}  {elapsed:.3f}s")

with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_8b, _encode_test_samples)

# Free model + reclaim memory
del emb_model_qwen3_emb_8b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")


#### 3.3.2 Extrinsic

##### 3.3.2.1a Embedding: reference PDDL vs generated PDDL Similarity

In [ ]:
# ── 3.3.2.1a Prepare extrinsic texts (shared across all embedding models) ──
extr_domains, extr_cve_ids, extr_labels, extr_groups = prepare_extrinsic_texts(dataset)
print(f"Extrinsic pairs prepared: {len(extr_domains)} domains, {len(extr_labels)} pairs")
print(f"  Positive: {extr_labels.sum()}, Negative: {(extr_labels == 0).sum()}")
print(f"  Groups (CVEs): {len(np.unique(extr_groups))}")


In [ ]:
# === 3.3.2.1a Embedding Model: Qwen/Qwen3-Embedding-8B ===
# Default settings (no max_seq cap, batch_size=8) + GPU OOM fallback to CPU.
if "emb_model_qwen3_emb_8b" not in dir():
    emb_model_qwen3_emb_8b = load_embedding_model("Qwen/Qwen3-Embedding-8B")
EMB_NAME_qwen3_emb_8b = "Qwen/Qwen3-Embedding-8B"

_bs = 8

# Step 1: Compute scores (OOM-safe)
def _compute_ext_scores_qwen3_emb_8b():
    _E = emb_model_qwen3_emb_8b.encode(extr_domains, convert_to_tensor=True, normalize_embeddings=True,
                                       show_progress_bar=True, batch_size=_bs)
    _sim_matrix = (_E @ _E.T).float().cpu().numpy()
    n_ext = len(extr_domains)
    return np.array([float(_sim_matrix[i, j]) for i in range(n_ext) for j in range(i+1, n_ext)])

t_build = time.time()
ext_scores_qwen3_emb_8b = with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_8b, _compute_ext_scores_qwen3_emb_8b)
ext_build_seconds_qwen3_emb_8b = round(time.time() - t_build, 2)
print(f"{EMB_NAME_qwen3_emb_8b}: build_pairs {ext_build_seconds_qwen3_emb_8b}s")

# Step 2: CV calibration
t_cv = time.time()
ext_threshold_qwen3_emb_8b, ext_fold_thr_qwen3_emb_8b, ext_cv_pred_qwen3_emb_8b, ext_cv_true_qwen3_emb_8b = run_calibration(
    ext_scores_qwen3_emb_8b, extr_labels, extr_groups)
ext_cv_seconds_qwen3_emb_8b = round(time.time() - t_cv, 2)
print(f"  CV calibration {ext_cv_seconds_qwen3_emb_8b}s, threshold={ext_threshold_qwen3_emb_8b:.4f}")

# Step 3: Performance report + save CV
ext_row_qwen3_emb_8b = report_row(ext_cv_true_qwen3_emb_8b, ext_cv_pred_qwen3_emb_8b,
    metric="embedding", model=EMB_NAME_qwen3_emb_8b, mode="extrinsic", threshold=ext_threshold_qwen3_emb_8b)
print(f"  TPR={ext_row_qwen3_emb_8b['tpr']:.4f}, FPR={ext_row_qwen3_emb_8b['fpr']:.4f}, F1={ext_row_qwen3_emb_8b.get('1__f1-score', 0):.4f}")
print(classification_report(ext_cv_true_qwen3_emb_8b, ext_cv_pred_qwen3_emb_8b, zero_division=0))

save_dir = os.path.join(RESULTS_BASE, "semantic", "extrinsic", "similarity")
os.makedirs(save_dir, exist_ok=True)
cv_path = os.path.join(save_dir, "cv_extrinsic_similarity.jsonl")
ext_cv_rec_qwen3_emb_8b = save_cv_record(cv_path, EMB_NAME_qwen3_emb_8b, ext_threshold_qwen3_emb_8b, ext_fold_thr_qwen3_emb_8b,
    ext_row_qwen3_emb_8b, extr_labels, ext_build_seconds_qwen3_emb_8b, ext_cv_seconds_qwen3_emb_8b)

calibration_embedding_rows.append({
    "model": EMB_NAME_qwen3_emb_8b, "mode": "extrinsic", "threshold": ext_threshold_qwen3_emb_8b,
    "precision": ext_cv_rec_qwen3_emb_8b["precision"], "recall": ext_cv_rec_qwen3_emb_8b["recall"],
    "f1": ext_cv_rec_qwen3_emb_8b["f1"], "tpr": ext_row_qwen3_emb_8b["tpr"], "fpr": ext_row_qwen3_emb_8b["fpr"],
    "n_positive": int(extr_labels.sum()), "n_negative": int((extr_labels == 0).sum()),
})

# Step 4: Test on test_samples (OOM-safe)
results_path = os.path.join(save_dir, "results_extrinsic_similarity.jsonl")
ref_by_cve = {entry["cve_id"]: entry["attack_paths"] for entry in dataset}
def _test_loop_ext_qwen3_emb_8b():
    print(f"\nApplying threshold {ext_threshold_qwen3_emb_8b:.4f} ({EMB_NAME_qwen3_emb_8b}) to test samples:")
    for source, cve_id, ap_id, domain_pddl in test_samples:
        ref_aps = ref_by_cve.get(cve_id, [])
        if not ref_aps:
            print(f"  [{source}] {cve_id}/{ap_id}  SKIP (no reference APs)")
            continue
        t0 = time.time()
        E_test = emb_model_qwen3_emb_8b.encode([domain_pddl], convert_to_tensor=True, normalize_embeddings=True)
        ref_texts = [ap["domain"] for ap in ref_aps]
        E_ref = emb_model_qwen3_emb_8b.encode(ref_texts, convert_to_tensor=True, normalize_embeddings=True)
        sims = (E_test @ E_ref.T).float().cpu().numpy()[0]
        elapsed = time.time() - t0
        best_sim = float(sims.max())
        best_js = [j for j in range(len(sims)) if float(sims[j]) == best_sim]
        pred = True if best_sim >= ext_threshold_qwen3_emb_8b else False
        for j, ref_ap in enumerate(ref_aps):
            sim = float(sims[j])
            p = True if sim >= ext_threshold_qwen3_emb_8b else False
            print(f"  [{source}] {cve_id}/{ap_id} vs {ref_ap['ap_id']}  sim={sim:.4f}  pred={p}  {elapsed:.3f}s")
        save_extrinsic_similarity_result(results_path, source, EMB_NAME_qwen3_emb_8b, ext_threshold_qwen3_emb_8b, cve_id, ap_id,
            [ref_aps[j]["ap_id"] for j in best_js], best_sim, pred, len(ref_aps), elapsed)
        print(f"  [{source}] {cve_id}/{ap_id} BEST={[ref_aps[j]['ap_id'] for j in best_js]}  sim={best_sim:.4f}  pred={pred}")

with_gpu_oom_cpu_fallback(emb_model_qwen3_emb_8b, _test_loop_ext_qwen3_emb_8b)

# Free model + reclaim memory
del emb_model_qwen3_emb_8b
import gc; gc.collect(); gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"  GPU free after release: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
